# Classical RAG: vectors, ranked evidence and answers

Follow one question from a page-based vector index to a source-grounded answer.
Classical RAG retrieves text by vector similarity and gives the ranked text to an
Azure OpenAI chat model. The answer prompt defines citations and abstention.

See the [Classical RAG workflow](../docs/illustrations/classical-rag-workflow.pdf)
from indexing to source-page evaluation.

**Prerequisites:** use the [README setup](../README.md) and read [Data and evidence](01_data_and_evidence.ipynb).
**Default run:** replay four illustrative RFC 2795 questions with saved Azure vectors
and answers, without credentials or new model requests. These are public example
results. The notebook checks stable ties and source assignment, then displays the
ranked pages, complete prompt context and unchanged answers.

Method details: thesis Section 2.5, Table 3 and Appendix C, Table 25.


The same execution path supports `C0`, `C-CHUNK-STRUCT-800-0`,
`C-EMBED-LARGE-1536` and `C-FINAL`. Select `CONDITION_ID` and the matching
prepared input or stored run folder. Structure segments retain their original
page Evidence Units and character intervals. See [controlled changes](04_controlled_changes.ipynb)
and [structure segmentation](04a_structure_chunking.ipynb) for the factor definitions.


In [1]:
import hashlib
import json
import os
import re
import shutil
from html import escape
from pathlib import Path

import numpy as np
from IPython.display import HTML, Markdown, display
from openai import AzureOpenAI
from tabulate import tabulate

from rag_comparison import config
from rag_comparison.embedding import embed_with_checkpoint

## 1. Select inputs and an execution mode

`data_dir` points to a manifest and its relative file references.
For **replay**, keep `RUN_AZURE = False` and select a stored run, including its matrix,
query vectors, retrievals and answers. This makes no model requests.
For **generation**, set `RUN_AZURE = True` and select page chunks, origins and questions;
these three input files and their manifest are sufficient. No earlier vectors or answers are required.
The generation section writes a separate `azure_dir` and can resume matching committed batches.

The same notebook uses public inputs or approved private files. Save private executed copies
under ignored `local/`. See the [artifact contract](../docs/classical_rag_artifacts.md).


In [2]:
CONDITION_ID = "C0"
condition = config.CONDITIONS[CONDITION_ID]
assert condition["system"] == "C"
embedding_model = condition["embedding_model"]
embedding_variable = config.EMBEDDING_DEPLOYMENT_VARIABLES[embedding_model]
data_dir = Path("../examples/classical_rag")
RUN_AZURE = False
azure_dir = Path("../local/classical_rag/azure-example")
manifest = json.loads((data_dir / "run_manifest.json").read_text(encoding="utf-8"))


def artifact_path(role: str) -> Path:
    reference = manifest["artifacts"][role]
    path = (data_dir / reference["path"]).resolve()
    assert path.is_relative_to(data_dir.resolve()), "Use paths inside the input folder."
    assert hashlib.sha256(path.read_bytes()).hexdigest() == reference["sha256"]
    return path


def read_records(role: str) -> list[dict]:
    return [
        json.loads(line)
        for line in artifact_path(role).read_text(encoding="utf-8").splitlines()
        if line
    ]

In [3]:
chunks = read_records("chunks")
origins = read_records("chunk_origins")
questions = read_records("questions")
question_by_id = {row["question_id"]: row for row in questions}
assert len(question_by_id) == len(questions)
if not RUN_AZURE:
    query_records = read_records("query_embeddings")
    retrieval_records = read_records("retrieval")
    answer_records = read_records("answers")
print(
    {
        "input_kind": manifest["artifact_kind"],
        "chunks": len(chunks),
        "questions": len(questions),
        "Azure_enabled": RUN_AZURE,
    }
)
assert (
    manifest.get("chunking_policy", config.PAGE_CHUNKING_POLICY)
    == condition["chunking_policy"]
)
if not RUN_AZURE:
    assert manifest["condition_id"] == CONDITION_ID

{'input_kind': 'azure_run', 'chunks': 20, 'questions': 4, 'Azure_enabled': False}


## 2. Bind matrix rows to source text

An embedding is a numeric representation of text. Document and question embeddings
use the same model and dimension. The baseline uses `text-embedding-3-small`, 1536
dimensions. The stored index contains unit-length Float32 rows. Its sidecar records
which chunk belongs to each row; changing that order can change score ties.

The public example contains 20 page chunks and actual Azure embeddings.
Each question is ranked against all 20 pages; the five highest-scoring pages enter the context.
Generation first checks source identity and creates its matrix later in the Azure section.

The [metric notebook](05_retrieval_metrics.ipynb) uses the saved five-place rankings
and the source-reviewed example relevance grades.


In [4]:
chunk_ids = [row["chunk_id"] for row in chunks]
assert len(chunk_ids) == len(set(chunk_ids))
assert chunk_ids == [row["chunk_id"] for row in origins]
for chunk, origin in zip(chunks, origins, strict=True):
    assert origin["projection_status"] == "projected"
    assert len(origin["origin_evidence_unit_ids"]) == 1
    assert origin["document_id"] == chunk["document_id"]
    assert origin["corpus_snapshot_id"] == chunk["corpus_snapshot_id"]
if not RUN_AZURE:
    matrix = np.load(artifact_path("index_matrix"), allow_pickle=False)
    sidecar = json.loads(artifact_path("index_sidecar").read_text(encoding="utf-8"))
    assert sidecar["item_ids"] == chunk_ids
    assert sidecar["dimension"] == config.EMBEDDING_DIMENSION
    assert sidecar["embedding_model"] == embedding_model
    assert matrix.dtype == np.float32
    assert matrix.shape == (len(chunks), config.EMBEDDING_DIMENSION)
    assert np.isfinite(matrix).all()
    assert np.allclose(np.linalg.norm(matrix, axis=1), 1.0, atol=1e-6)
    print({"shape": matrix.shape, "dtype": str(matrix.dtype), "unit_length_rows": True})

{'shape': (20, 1536), 'dtype': 'float32', 'unit_length_rows': True}


## 3. Exact cosine similarity and stable ties

Cosine similarity measures vector direction: $\cos(x,q)=(x\cdot q)/(\|x\|\|q\|)$.
After normalising both vectors, the dot product gives the same value.
NumPy calculates every score; there is no approximate nearest-neighbour search.
`argsort(..., kind="stable")` keeps matrix-row order when scores are equal.
The small calculation below has expected order **0, 2, 1**.

In [5]:
toy_matrix = np.asarray([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0]], dtype=np.float32)
toy_query = np.asarray([1.0, 0.0], dtype=np.float32)
dot_scores = toy_matrix @ toy_query
manual_scores = [
    np.dot(row, toy_query) / (np.linalg.norm(row) * np.linalg.norm(toy_query))
    for row in toy_matrix
]
np.testing.assert_allclose(dot_scores, manual_scores)
stable_order = np.argsort(-dot_scores, kind="stable")
assert stable_order.tolist() == [0, 2, 1]
print({"scores": dot_scores.tolist(), "stable_row_order": stable_order.tolist()})

{'scores': [1.0, 0.0, 1.0], 'stable_row_order': [0, 2, 1]}


## 4. Replay ranked evidence and build the full context

The helper below keeps the calculation and source assignment together. It uses the
currently selected matrix and page records, so the public and private paths share
the same steps. Query vectors are normalised before scoring. A zero or nonfinite
vector is an invalid input, not a valid low-scoring question.

Each hit keeps its rank, score, source page and Evidence Unit. The context includes
every selected text in full and in rank order, with German source headers `[n]`.

Replay requires identical ranks, source IDs, texts and contexts. Float32 score
recalculations use an absolute tolerance of 1e-6; saved scores remain unchanged.

In [6]:
def retrieve(query_vector: list[float], index: np.ndarray) -> tuple[list[dict], str]:
    vector = np.asarray(query_vector, dtype=np.float32)
    assert vector.shape == (config.EMBEDDING_DIMENSION,)
    norm = np.linalg.norm(vector)
    assert np.isfinite(vector).all() and np.isfinite(norm) and norm > 0
    scores = index @ (vector / norm)
    order = np.argsort(-scores, kind="stable")[: config.TOP_K]
    hits = []
    for rank, row_index in enumerate(order, start=1):
        chunk, origin = chunks[int(row_index)], origins[int(row_index)]
        hits.append(
            {
                "rank": rank,
                "score": float(scores[int(row_index)]),
                "chunk_id": chunk["chunk_id"],
                "evidence_unit_id": origin["origin_evidence_unit_ids"][0],
                "projection_status": origin["projection_status"],
                "document_id": chunk["document_id"],
                "page": chunk["page_number"],
                "text": chunk["text"],
            }
        )
    for hit, row_index in zip(hits, order, strict=True):
        chunk = chunks[int(row_index)]
        hit.update(
            {
                field: chunk[field]
                for field in (
                    "source_page_chunk_id",
                    "character_start",
                    "character_end",
                )
                if field in chunk
            }
        )
    context = "\n\n".join(
        f"[{hit['rank']}] (Dokument {hit['document_id']}, Seite {hit['page']}): {hit['text']}"
        for hit in hits
    )
    return hits, context

In [7]:
if not RUN_AZURE:
    query_by_id = {row["question_id"]: row for row in query_records}
    assert len(query_by_id) == len(query_records)
    replayed = []
    for record in retrieval_records:
        query = query_by_id[record["question_id"]]
        for field in (
            "system_id",
            "run_id",
            "configuration_sha256",
            "question_set_version",
        ):
            assert record[field] == query[field] == manifest[field]
        assert (
            query["question_text"]
            == question_by_id[record["question_id"]]["question_text"]
        )
        if record["technical_status"] == "completed":
            assert query["technical_status"] == "completed"
            assert (
                record["index_sha256"]
                == manifest["artifacts"]["index_matrix"]["sha256"]
            )
            hits, context = retrieve(query["vector"], matrix)
            assert context == record["context"]
            assert [{k: v for k, v in hit.items() if k != "score"} for hit in hits] == [
                {k: v for k, v in hit.items() if k != "score"} for hit in record["hits"]
            ]
            np.testing.assert_allclose(
                [hit["score"] for hit in hits],
                [hit["score"] for hit in record["hits"]],
                rtol=0,
                atol=1e-6,
            )
            replayed.append((record["question_id"], [h["page"] for h in hits]))
    print({"replayed_rankings": len(replayed), "first_page_orders": replayed[:2]})
    print(retrieval_records[0]["context"][:200] + " …")

{'replayed_rankings': 4, 'first_page_orders': [('DEMO-IMPS-Q-001', [8, 7, 9, 17, 10]), ('DEMO-IMPS-Q-002', [12, 13, 3, 16, 14])]}
[1] (Dokument RFC-2795, Seite 8): RFC 2795 The Infinite Monkey Protocol Suite (IMPS) 1 April 2000 5.2 KEEPER Message Response Codes (SIMIAN-to-ZOO) CODE NAME DESCRIPTION +----------------------------- …


## 5. Understand the answer contract

The chat model answers in German, uses only the supplied context, cites factual
claims with `[n]`, and ends with a `Quellen:` line containing the used markers.
If the context is insufficient, the entire answer must be the fixed abstention sentence,
without a sources line. These are formatting rules; answer quality requires the separate
evaluation station. A response can satisfy this format and still be factually wrong.

The following small checker also detects invalid or mismatched source markers.
It never rewrites an answer or requests a replacement.

In [8]:
CITATION = re.compile(r"\[(\d+)\]")
SOURCE_LINE = re.compile(r"^Quellen:\s*(.*)$")
MARKS_ONLY = re.compile(r"(?:\[\d+\](?:\s*,?\s*)?)+")


def inspect_answer(text: str, hits: list[dict]) -> dict:
    """Check source-marker format; this does not judge answer quality."""
    stripped = text.strip()
    if stripped == config.GENERATION_ABSTENTION_MARKER:
        return {
            "answer_behavior": "abstains",
            "answer_contract_status": "valid",
            "cited_evidence_unit_ids": [],
            "invalid_marks": [],
        }
    lines = stripped.splitlines()
    source_positions = [
        index for index, line in enumerate(lines) if SOURCE_LINE.fullmatch(line)
    ]
    if len(source_positions) == 1:
        source_index = source_positions[0]
        source_payload = SOURCE_LINE.fullmatch(lines[source_index]).group(1).strip()
    else:
        source_index = -1
        source_payload = ""
    body = "\n".join(lines[:source_index]).strip() if source_index >= 0 else stripped
    body_marks = {int(mark) for mark in CITATION.findall(body)}
    source_marks = {int(mark) for mark in CITATION.findall(source_payload)}
    invalid = sorted(
        mark for mark in body_marks | source_marks if not 1 <= mark <= len(hits)
    )
    behavior = (
        "answers_with_insufficient_caveat"
        if config.GENERATION_ABSTENTION_MARKER in stripped
        else "answers"
    )
    valid = (
        behavior == "answers"
        and bool(body)
        and bool(body_marks)
        and len(source_positions) == 1
        and source_index == len(lines) - 1
        and bool(MARKS_ONLY.fullmatch(source_payload))
        and body_marks == source_marks
        and not invalid
    )
    cited = [
        hits[mark - 1]["evidence_unit_id"]
        for mark in sorted(body_marks)
        if 1 <= mark <= len(hits)
    ]
    return {
        "answer_behavior": behavior,
        "answer_contract_status": "valid" if valid else "invalid",
        "cited_evidence_unit_ids": cited,
        "invalid_marks": invalid,
    }

In [9]:
if not RUN_AZURE:
    retrieval_by_id = {row["question_id"]: row for row in retrieval_records}
    assert len(retrieval_by_id) == len(retrieval_records)
    assert len({row["question_id"] for row in answer_records}) == len(answer_records)
    rows = []
    for answer in answer_records:
        assert answer["question_id"] in question_by_id
        for field in (
            "system_id",
            "run_id",
            "configuration_sha256",
            "question_set_version",
        ):
            assert answer[field] == manifest[field]
        if answer["technical_status"] == "completed":
            hits = retrieval_by_id[answer["question_id"]]["hits"]
            observed = inspect_answer(answer["answer_text"], hits)
            assert all(answer[key] == value for key, value in observed.items())
            rows.append(
                (
                    answer["question_id"],
                    observed["answer_behavior"],
                    observed["answer_contract_status"],
                )
            )

## 6. Generate with Azure OpenAI when explicitly selected

This optional section uses the official `AzureOpenAI` client. Copy `.env.example`
to `.env`, fill the Azure values, and use the
[Azure setup command](../docs/usage.md#configure-azure-for-the-pipeline-steps).
Deployment names select your deployed `text-embedding-3-small` and `gpt-4.1-mini` models.
Use an API version supported by those deployments.

Set `RUN_AZURE = True` only when you intend to make paid requests. Inputs and outputs
are saved in ignored `local/`; completed embedding batches and answers are reused.
The SDK allows at most one retry for transient errors (two attempts total).
Errors remain visible; an interrupted directory is incomplete until this section finishes.
The seed provides best-effort repeatability, not guaranteed identical new answers.

The output directory is also bound to endpoint, API version and deployment names.
Saved answers must match their exact retrieval file before they are reused.

In [10]:
parameters = {
    "embedding_model": embedding_model,
    "dimension": config.EMBEDDING_DIMENSION,
    "top_k": config.TOP_K,
    "generation_model": config.GENERATION_MODEL,
    "generation_decoding": config.GENERATION_DECODING,
    "system_prompt": config.GENERATION_SYSTEM_PROMPT,
    "user_template": config.GENERATION_USER_TEMPLATE,
}
print(
    {
        key: value
        for key, value in parameters.items()
        if key not in ("system_prompt", "user_template")
    }
)

{'embedding_model': 'text-embedding-3-small', 'dimension': 1536, 'top_k': 5, 'generation_model': 'gpt-4.1-mini', 'generation_decoding': {'temperature': 0, 'top_p': 1, 'max_tokens': 1024, 'seed': 42}}


In [11]:
def write_records(path: Path, records: list[dict]) -> None:
    content = "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in records)
    if path.exists():
        assert path.read_text(encoding="utf-8") == content, (
            "Use a new directory for changed results."
        )
    else:
        path.write_text(content, encoding="utf-8")


if RUN_AZURE:
    assert azure_dir.resolve().is_relative_to(Path("../local").resolve())
    azure_dir.mkdir(parents=True, exist_ok=True)
    azure_binding = {
        name: os.environ[name]
        for name in (
            "AZURE_OPENAI_ENDPOINT",
            "AZURE_OPENAI_API_VERSION",
            embedding_variable,
            "AZURE_OPENAI_CHAT_DEPLOYMENT",
        )
    }
    azure_parameters = {
        **parameters,
        "azure_binding_sha256": hashlib.sha256(
            json.dumps(azure_binding, sort_keys=True).encode("utf-8")
        ).hexdigest(),
    }
    configuration_text = (
        json.dumps(azure_parameters, ensure_ascii=False, indent=2) + "\n"
    )
    configuration_path = azure_dir / "configuration.json"
    if configuration_path.exists():
        assert configuration_path.read_text(encoding="utf-8") == configuration_text
    configuration_path.write_text(configuration_text, encoding="utf-8")

In [12]:
if RUN_AZURE:
    for role in ("chunks", "chunk_origins", "questions"):
        target = azure_dir / f"{role}.jsonl"
        source = artifact_path(role)
        if target.exists():
            assert target.read_bytes() == source.read_bytes()
        shutil.copyfile(source, target)
    client = AzureOpenAI(
        api_key=os.environ["AZURE_OPENAI_API_KEY"],
        azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
        api_version=os.environ["AZURE_OPENAI_API_VERSION"],
        max_retries=config.PROVIDER_MAX_ATTEMPTS - 1,
    )

### Embed and checkpoint the selected text

The direct SDK call requests 1536 dimensions. Response indices restore input order.
The checkpoint helper only stores committed Float32 batches and their progress;
it binds ordered IDs, exact text hashes, model, dimension and batch size.
Changing a bound input requires a separate output directory. A completed checkpoint
returns its stored vectors without another embedding request.

The document index can also be supplied as existing matrix and sidecar files, as in
the offline section. There is no need to regenerate embeddings for replay or export.

In [13]:
if RUN_AZURE:

    def embed_texts(texts: list[str]) -> list[list[float]]:
        response = client.embeddings.create(
            model=os.environ[embedding_variable],
            input=texts,
            dimensions=config.EMBEDDING_DIMENSION,
        )
        assert response.model == embedding_model
        assert sorted(item.index for item in response.data) == list(range(len(texts)))
        return [
            item.embedding
            for item in sorted(response.data, key=lambda item: item.index)
        ]

In [14]:
if RUN_AZURE:
    document_vectors = embed_with_checkpoint(
        chunk_ids,
        [row["text"] for row in chunks],
        embed_texts,
        azure_dir / "document_batches",
        model=embedding_model,
    )
    query_vectors = embed_with_checkpoint(
        [row["question_id"] for row in questions],
        [row["question_text"] for row in questions],
        embed_texts,
        azure_dir / "query_batches",
        model=embedding_model,
    )
    azure_matrix = document_vectors / np.linalg.norm(
        document_vectors, axis=1, keepdims=True
    )
    (azure_dir / "index").mkdir(exist_ok=True)
    np.save(azure_dir / "index/embedding_matrix.npy", azure_matrix, allow_pickle=False)
    azure_sidecar = {
        "run_id": azure_dir.name,
        "embedding_model": embedding_model,
        "dimension": config.EMBEDDING_DIMENSION,
        "item_ids": chunk_ids,
        "index_family": "numpy-exact-cosine",
        "normalization_status": "normalized_unit_length",
    }
    (azure_dir / "index/index_sidecar.json").write_text(
        json.dumps(azure_sidecar, indent=2) + "\n", encoding="utf-8"
    )

In [15]:
if RUN_AZURE:
    run_binding = {
        "system_id": "CLASSICAL-RAG",
        "condition_id": CONDITION_ID,
        "run_id": azure_dir.name,
        "configuration_sha256": hashlib.sha256(
            configuration_path.read_bytes()
        ).hexdigest(),
        "question_set_version": manifest["question_set_version"],
    }
    generated_queries, generated_retrievals = [], []
    index_sha = hashlib.sha256(
        (azure_dir / "index/embedding_matrix.npy").read_bytes()
    ).hexdigest()
    for question, vector in zip(questions, query_vectors, strict=True):
        generated_queries.append(
            {
                **run_binding,
                **question,
                "technical_status": "completed",
                "embedding_model": embedding_model,
                "vector": vector.tolist(),
            }
        )
        hits, context = retrieve(vector.tolist(), azure_matrix)
        generated_retrievals.append(
            {
                **run_binding,
                **question,
                "technical_status": "completed",
                "index_sha256": index_sha,
                "hits": hits,
                "context": context,
            }
        )
    write_records(azure_dir / "query_embeddings.jsonl", generated_queries)
    write_records(azure_dir / "retrieval.jsonl", generated_retrievals)

### Send the full ranked context and retain the response

The system and user prompts keep their German wording. The request uses temperature 0,
top-p 1, seed 42 and at most 1024 output tokens. It neither truncates retrieved page text
nor repairs a response. The returned model family must match `gpt-4.1-mini`.
Responses retain their actual model identifier, token usage and source-format status.
Completed saved answers are skipped on a repeated run.

In [16]:
if RUN_AZURE:
    answers_path = azure_dir / "answers.jsonl"
    generated_answers = (
        [
            json.loads(line)
            for line in answers_path.read_text(encoding="utf-8").splitlines()
            if line
        ]
        if answers_path.exists()
        else []
    )
    completed_ids = {row["question_id"] for row in generated_answers}
    assert len(completed_ids) == len(generated_answers)
    assert completed_ids <= {row["question_id"] for row in questions}
    retrieval_sha = hashlib.sha256(
        (azure_dir / "retrieval.jsonl").read_bytes()
    ).hexdigest()
    for saved in generated_answers:
        assert saved["technical_status"] == "completed"
        assert saved["retrieval_sha256"] == retrieval_sha
        assert all(saved[field] == value for field, value in run_binding.items())

In [17]:
if RUN_AZURE:
    for record in generated_retrievals:
        if record["question_id"] in completed_ids:
            continue
        response = client.chat.completions.create(
            model=os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"],
            messages=[
                {"role": "system", "content": config.GENERATION_SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": config.GENERATION_USER_TEMPLATE.format(
                        context=record["context"], question=record["question_text"]
                    ),
                },
            ],
            **config.GENERATION_DECODING,
        )
        assert (
            re.sub(r"-\d{4}-\d{2}-\d{2}$", "", response.model)
            == config.GENERATION_MODEL
        )
        answer_text = response.choices[0].message.content or ""
        usage = response.usage
        answer = {
            **run_binding,
            "question_id": record["question_id"],
            "technical_status": "completed",
            "retrieval_sha256": retrieval_sha,
            "generation_model": response.model,
            "system_fingerprint": response.system_fingerprint,
            "prompt_tokens": usage.prompt_tokens if usage else None,
            "completion_tokens": usage.completion_tokens if usage else None,
            "answer_text": answer_text,
            **inspect_answer(answer_text, record["hits"]),
        }
        with answers_path.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(answer, ensure_ascii=False) + "\n")
        generated_answers.append(answer)

### Bind the completed files

The run manifest records relative paths and SHA256 hashes. These files can then be read
with `RUN_AZURE = False` using the same cells above. No Azure call is needed to inspect
them, calculate metrics, or export the selected stored artefacts.
Only a complete generation receives a completed manifest; interrupted work stays
in the ignored output directory and can be resumed there.

In [18]:
if RUN_AZURE:
    assert {row["question_id"] for row in generated_answers} == set(question_by_id)
    assert len(generated_answers) == len(questions)
    paths = {
        role: f"{role}.jsonl"
        for role in (
            "chunks",
            "chunk_origins",
            "questions",
            "query_embeddings",
            "retrieval",
            "answers",
        )
    }
    paths.update(
        configuration="configuration.json",
        index_matrix="index/embedding_matrix.npy",
        index_sidecar="index/index_sidecar.json",
    )
    generated_manifest = {
        "schema_version": "classical-rag-artifacts-v1",
        "artifact_kind": "azure_run",
        "status": "completed",
        "chunking_policy": condition["chunking_policy"],
        **run_binding,
        "artifacts": {
            role: {
                "path": path,
                "sha256": hashlib.sha256((azure_dir / path).read_bytes()).hexdigest(),
            }
            for role, path in paths.items()
        },
    }
    (azure_dir / "run_manifest.json").write_text(
        json.dumps(generated_manifest, indent=2) + "\n", encoding="utf-8"
    )
    print({"saved_answers": len(generated_answers), "status": "completed"})
else:
    print("Offline replay complete. No Azure client or model request was created.")

Offline replay complete. No Azure client or model request was created.


## Retrieval and response records

The table reports retrieval ranks and cosine scores for the selected question.
The expandable output contains the complete user-prompt context. The system prompt
in [`config.py`](../src/rag_comparison/config.py) defines the German citation and
abstention rules. The responses below are unchanged saved outputs.

In [19]:
selected_retrievals = generated_retrievals if RUN_AZURE else retrieval_records
selected_answers = generated_answers if RUN_AZURE else answer_records
answers_by_id = {row["question_id"]: row for row in selected_answers}
question = questions[0]
retrieval = next(
    row for row in selected_retrievals if row["question_id"] == question["question_id"]
)
answer = answers_by_id[question["question_id"]]
display(Markdown(f"**Question:** {question['question_text']}"))
source_rows = [
    (hit["rank"], hit["document_id"], hit["page"], f"{hit['score']:.6f}")
    for hit in retrieval["hits"]
]
display(
    Markdown(
        tabulate(
            source_rows,
            headers=["Rank", "Document", "Page", "Cosine score"],
            tablefmt="github",
        )
    )
)
display(
    HTML(
        "<details><summary>Complete context sent to the model</summary>"
        "<pre style='white-space: pre-wrap; word-break: normal'>"
        + escape(retrieval["context"])
        + "</pre></details>"
    )
)

**Question:** Which numeric KEEPER request code stops all monkey business, which response codes may a SIMIAN use, and how can an attacker misuse that request?

|   Rank | Document   |   Page |   Cosine score |
|--------|------------|--------|----------------|
|      1 | RFC-2795   |      8 |       0.736116 |
|      2 | RFC-2795   |      7 |       0.689466 |
|      3 | RFC-2795   |      9 |       0.622338 |
|      4 | RFC-2795   |     17 |       0.596859 |
|      5 | RFC-2795   |     10 |       0.595749 |

In [20]:
format_rows = [
    (row["question_id"], row["answer_behavior"], row["answer_contract_status"])
    for row in selected_answers
    if row["technical_status"] == "completed"
]
display(
    Markdown(
        tabulate(
            format_rows[:5],
            headers=["Question", "Behaviour", "Source format"],
            tablefmt="github",
        )
    )
)
display(Markdown(f"**Actual saved answer:**\n\n{answer['answer_text']}"))
print(
    {
        "returned_model": answer["generation_model"],
        "behaviour": answer["answer_behavior"],
        "format": answer["answer_contract_status"],
    }
)
unanswerable = next(
    (q for q in questions if q.get("cohort") == "robustness_abstention"), None
)
if unanswerable is not None:
    abstention_answer = answers_by_id[unanswerable["question_id"]]
    display(
        Markdown(
            f"**Question with missing source information:** {unanswerable['question_text']}\n\n"
            f"**Actual saved answer:** {abstention_answer['answer_text']}"
        )
    )

| Question        | Behaviour   | Source format   |
|-----------------|-------------|-----------------|
| DEMO-IMPS-Q-001 | answers     | valid           |
| DEMO-IMPS-Q-002 | answers     | invalid         |
| DEMO-IMPS-Q-003 | answers     | valid           |
| DEMO-IMPS-Q-004 | abstains    | valid           |

**Actual saved answer:**

Der numerische KEEPER Request-Code, der alle Affengeschäfte stoppt, ist 7 (STOP) [2]. Ein SIMIAN muss auf eine STOP-Anfrage mit einem der folgenden Codes antworten: NORESPONSE, ALIVE, DEAD oder GONE [1]. Ein Angreifer könnte diesen STOP-Request missbrauchen, indem er falsche Informationen wie den Status DEAD sendet, was dazu führen könnte, dass ein ZOO versucht, einen noch funktionierenden Affen zu ersetzen [4].

Quellen: [1], [2], [4]

{'returned_model': 'gpt-4.1-mini-2025-04-14', 'behaviour': 'answers', 'format': 'valid'}


**Question with missing source information:** What response timeout, in milliseconds, does CHIMP require for a SEND FOOD request?

**Actual saved answer:** Diese Frage lässt sich mit den vorliegenden Richtlinien nicht beantworten.